In [1]:
# Suggested data science package imports
import pandas as pd
import numpy as np

from datetime import datetime
import matplotlib
import matplotlib.pyplot as plt

# TrendMiner package import
import trendminer
from trendminer.trendminer_client import TrendMinerClient

import os

token = os.environ["KERNEL_USER_TOKEN"]

# Create TrendMiner API object
client = TrendMinerClient(token)

In [2]:
# Loading ContextHub view: OEE Monthly
from trendminer.context.context_view import ContextView
import uuid

context_view = ContextView(client, uuid.UUID("41df698f-2d29-4b67-9c88-c3d99c59ca95"))
df_month = context_view.load_view()

In [3]:
# --- MULTI-LINE: OEE and components over time ---

import plotly.graph_objects as go

# Ensure dates are datetime and sort chronologically
dfp = df_month.copy()
dfp["start_date"] = pd.to_datetime(dfp["start_date"])
dfp = dfp.sort_values("start_date")

# Optional: ensure metrics are in 0..1 (if your data is 0..100, divide by 100)
# for col in ["OEE_availability","OEE_performance","OEE_quality","OEE_oee_score"]:
#     if dfp[col].max() > 1.5:  # quick sanity check
#         dfp[col] = dfp[col] / 100.0

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=dfp["start_date"], y=dfp["OEE_oee_score"], mode="lines+markers", name="OEE"
))
fig.add_trace(go.Scatter(
    x=dfp["start_date"], y=dfp["OEE_availability"], mode="lines+markers", name="Availability"
))
fig.add_trace(go.Scatter(
    x=dfp["start_date"], y=dfp["OEE_performance"], mode="lines+markers", name="Performance"
))
fig.add_trace(go.Scatter(
    x=dfp["start_date"], y=dfp["OEE_quality"], mode="lines+markers", name="Quality"
))

fig.update_layout(
    title="OEE and Components Over Time",
    xaxis_title="Month",
    yaxis_title="Score (0–1)",
    hovermode="x unified",
)
fig.show()


In [4]:
# --- STACKED BAR: Loss contributions per month (1 - A, 1 - P, 1 - Q) ---

import plotly.graph_objects as go

dfs = df_month.copy()
dfs["start_date"] = pd.to_datetime(dfs["start_date"])
dfs = dfs.sort_values("start_date")

# Compute component losses (assuming metrics are in 0..1)
dfs["Loss_A"] = 1.0 - dfs["OEE_availability"]
dfs["Loss_P"] = 1.0 - dfs["OEE_performance"]
dfs["Loss_Q"] = 1.0 - dfs["OEE_quality"]

# Optional: label for x-axis
dfs["month_label"] = dfs["start_date"].dt.strftime("%Y-%m")

fig = go.Figure()
fig.add_bar(x=dfs["month_label"], y=dfs["Loss_A"], name="Availability Loss")
fig.add_bar(x=dfs["month_label"], y=dfs["Loss_P"], name="Performance Loss")
fig.add_bar(x=dfs["month_label"], y=dfs["Loss_Q"], name="Quality Loss")

fig.update_layout(
    barmode="stack",
    title="Loss Contributions by Component (per Month)",
    xaxis_title="Month",
    yaxis_title="Loss (fraction of 1.0)",
    hovermode="x unified",
)
fig.show()


In [3]:
# --- RADAR: A/P/Q for the last N months ---

import plotly.graph_objects as go

LAST_N = 4  # change to show more/fewer months

dfr = df_month.copy()
dfr["start_date"] = pd.to_datetime(dfr["start_date"])
dfr = dfr.sort_values("start_date").tail(LAST_N)

# Build a radar (polar) chart with one trace per month
categories = ["OEE_availability", "OEE_performance", "OEE_quality"]
cat_labels = ["Availability", "Performance", "Quality"]

fig = go.Figure()

for _, row in dfr.iterrows():
    values = [row[c] for c in categories]
    # Close the polygon by repeating the first point
    values_closed = values + values[:1]
    fig.add_trace(go.Scatterpolar(
        r=values_closed,
        theta=cat_labels + cat_labels[:1],
        name=row["start_date"].strftime("%Y-%m"),
        mode="lines+markers",
        fill="toself",
    ))

fig.update_layout(
    title=f"Radar of A/P/Q for Last {len(dfr)} Months",
    polar=dict(
        radialaxis=dict(range=[0, 1], tickformat=".0%", showline=True),
    ),
    showlegend=True,
)
fig.show()
